In [15]:
pip install openjij

In [16]:
import pandas as pd
import numpy as np
import openjij as oj
import os

In [21]:
# === 1. ファイル読み込み ===
# ファイル設定
file_path = "山陽IC_INPUT.xlsx"

if not os.path.exists(file_path):
    print(f"エラー: '{file_path}' が見つかりません。")
else:
    df = pd.read_excel(file_path)

    # B列: 荷物ナンバー / C列: 配送先
    item_ids_raw = df.iloc[:, 1].dropna().astype(str).tolist()
    destinations_raw = df.iloc[:, 2].dropna().astype(str).tolist()

    # ヘッダー行や数値以外のデータを除外してクレンジング
    valid_indices = [idx for idx, val in enumerate(item_ids_raw) if val.isdigit()]
    item_ids = [item_ids_raw[i] for i in valid_indices]
    destinations = [destinations_raw[i] for i in valid_indices]

    n_items = len(destinations)
    print(f"✅ Section 1 完了: 荷物 {n_items} 件を正常に読み込みました。")

✅ Section 1 完了: 荷物 93 件を正常に読み込みました。


/usr/local/lib/python3.12/dist-packages/openpyxl/worksheet/header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")


In [23]:
# === 2. QUBO構築 ===
# 設定パラメータ
n_trucks = 20
n_slots = 6
n_variables = n_items * n_trucks * n_slots

lam_unique = 20000  # 重複禁止（絶対）
lam_cluster = 1500  # 行き先集約報酬
lam_truck_cost = 300 # 台数削減コスト

def get_idx(i, v, p):
    return i * (n_trucks * n_slots) + v * n_slots + p

QUBO = {}
def add_qubo(i, j, val):
    if i > j: i, j = j, i
    QUBO[(i, j)] = QUBO.get((i, j), 0) + val

# (A) 一意制約：各荷物は1回だけ
for i in range(n_items):
    indices = [get_idx(i, v, p) for v in range(n_trucks) for p in range(n_slots)]
    for idx1 in indices:
        add_qubo(idx1, idx1, -1 * lam_unique)
        for idx2 in indices:
            if idx1 >= idx2: continue
            add_qubo(idx1, idx2, 2 * lam_unique)

# (B) 同一目的地集約
for v in range(n_trucks):
    indices_v = [get_idx(i, v, p) for i in range(n_items) for p in range(n_slots)]
    for idx1 in indices_v:
        for idx2 in indices_v:
            if idx1 >= idx2: continue
            i1, i2 = idx1 // (n_trucks * n_slots), idx2 // (n_trucks * n_slots)

            if destinations[i1] == destinations[i2]:
                add_qubo(idx1, idx2, -lam_cluster)
            else:
                add_qubo(idx1, idx2, -lam_truck_cost * 0.1)

print(f"✅ Section 2 完了: QUBO行列を構築しました。変数個数: {n_variables}")

✅ Section 2 完了: QUBO行列を構築しました。変数個数: 11160


In [24]:
 # === 3. 実行 ===
print(" 最適化計算を実行中...")
sampler = oj.SASampler()
# num_readsを増やすと精度が上がります
result = sampler.sample_qubo(QUBO, num_reads=50)
best_solution = result.record[0][0]

print("✅ Section 3 完了: 計算が終了しました。")

🚀 最適化計算を実行中... 数秒かかります。
✅ Section 3 完了: 計算が終了しました。


In [25]:
# === 4. 結果表示 ===
print(f"--- 運行計画（行き先集約・No.順ソート） ---\n")
used_trucks = 0
total_loaded = 0

for v in range(n_trucks):
    truck_items = []
    for p in range(n_slots):
        for i in range(n_items):
            if best_solution[get_idx(i, v, p)] == 1:
                truck_items.append((int(item_ids[i]), destinations[i]))
                total_loaded += 1

    if truck_items:
        used_trucks += 1
        # 荷物No.で昇順ソート
        truck_items.sort(key=lambda x: x[0])

        # 整形して表示
        display_list = [f"No.{item[0]}({item[1]})" for item in truck_items]
        print(f"トラック {used_trucks:02d} [{len(truck_items)}台]: " + " | ".join(display_list))

print(f"\n✅ Section 4 完了: 全{n_items}件中、{total_loaded}件を {used_trucks}台に集約しました。")

--- 運行計画（行き先集約・No.順ソート） ---

トラック 01 [4台]: No.74(○○倉敷) | No.81(△△車輛) | No.82(△△車輛) | No.87(○○岡山津山店)
トラック 02 [4台]: No.6(高野店 → 中古車C) | No.58(野田店 → 中古車C) | No.76(○○岡山青江店) | No.103(備前店)
トラック 03 [7台]: No.3(中古車C → 高野店) | No.5(高野店) | No.9(高野店 → 中古車C) | No.50(野田店) | No.52(野田店 → 高野店) | No.68(○○岡山) | No.80(○○車輛)
トラック 04 [2台]: No.13(津山B) | No.55(野田店 → 中古車C)
トラック 05 [5台]: No.4(中古車C → 高野店) | No.14(津山B → 中古車C) | No.16(津山B → 中古車C) | No.35(十日市店（岡山B）) | No.36(十日市店（岡山B）)
トラック 06 [2台]: No.56(野田店 → 中古車C) | No.73(○○会社)
トラック 07 [7台]: No.2(真庭B → 中古車C) | No.7(高野店 → 中古車C) | No.12(津山B) | No.26(倉敷中央店) | No.46(十日市店 → 中古車C) | No.61(野田店（法人）) | No.72(○○オートセンター)
トラック 08 [1台]: No.38(高屋店 → 中古車C)
トラック 09 [5台]: No.24(吉備路店 → 中古車C) | No.25(吉備路店 → 中古車C) | No.54(野田店 → 中古車C) | No.57(野田店 → 中古車C) | No.79(○○車輛)
トラック 10 [7台]: No.19(水島店) | No.23(吉備路店) | No.42(十日市店) | No.53(野田店 → 中古車C) | No.83(○○岡山営業所) | No.84(○○岡山営業所) | No.104(備前店)
トラック 11 [4台]: No.8(高野店 → 中古車C) | No.29(倉敷中島店) | No.69(○○東岡山) | No.75(○○倉敷)
トラック 12 [4台]: No.15(津山B →